### Importing modules

In [1]:
import re

### Loading gray fox reference genome

In [2]:
%%time
path_to_genome = '../data/ref_genome/GCA_032313775.1_UCinereo1.0_genomic.fna'
genome = open(path_to_genome, 'r')
genome = ''.join([line.rstrip() for line in genome])

CPU times: user 7.12 s, sys: 4.95 s, total: 12.1 s
Wall time: 11.1 s


In [3]:
print(len(genome))

2658882457


### Searching for target sequence

In [4]:
target_regex = 'GCA.{6}TCG|CGA.{6}TGC' 

In [5]:
pattern = re.compile(target_regex, re.IGNORECASE)
matches = re.findall(pattern, genome)

In [6]:
print(f'''{len(matches)} instances of the {target_regex} pattern covering 
{len(matches) * 36} base pairs or {round(len(matches)*36/len(genome)*100, 4)}% of the genome''')

178698 instances of the GCA.{6}TCG|CGA.{6}TGC pattern covering 
6433128 base pairs or 0.2419% of the genome


### Searching for target sequence (w/ adapters)

In [7]:
target_regex_adp = '.[GA].{10}GCA.{6}TCG.{10}C.|.[GA].{10}CGA.{6}TGC.{10}C.' 

In [8]:
pattern_adp = re.compile(target_regex_adp, re.IGNORECASE)
matches_adp = re.findall(pattern_adp, genome)

In [9]:
print(f'''{len(matches_adp)} instances of the {target_regex_adp} pattern covering 
{len(matches_adp) * 36} base pairs or {round(len(matches_adp)*36/len(genome)*100, 4)}% of the genome''')

25220 instances of the .[GA].{10}GCA.{6}TCG.{10}C.|.[GA].{10}CGA.{6}TGC.{10}C. pattern covering 
907920 base pairs or 0.0341% of the genome


### Making fastq file with our target sequence

In [10]:
with open('../data/ref_genome/bcg1_sites.txt', 'w') as f:
    for count, match in enumerate(matches_adp):
        f.write(f'@J00102:28:HTWWLBBXX:1:1101:{count}:1103 1:N:0:NCTTGA bcd=TGTC \n')
        f.write(match.upper() + '\n')
        f.write('+ \n')
        f.write(len(match)*'J' + '\n')
f.close()

### Aligning bcg1_sites to reference genome

In [11]:
%%bash
bowtie2 --no-unal --score-min L,16,1 --local -L 16 -x ../data/ref_genome/UCinero_ref -U ../data/ref_genome/bcg1_sites.txt -S ../data/ref_genome/bcg1_sites.bam

25220 reads; of these:
  25220 (100.00%) were unpaired; of these:
    0 (0.00%) aligned 0 times
    17139 (67.96%) aligned exactly 1 time
    8081 (32.04%) aligned >1 times
100.00% overall alignment rate
